###Adding, Removing, and Renaming Columns
----------------------------------------------------------------------------------------------------------------------------------------------

####1. Create a dataframe as below
 ```
   +---+--------+---+------+
   | id|    name|age|salary|
   +---+--------+---+------+
   |100|Prashant| 45| 45000|
   |101|   Tarun| 36| 33000|
   |102|   David| 48| 28000|
   +---+--------+---+------+
 ```

In [0]:
schema = "id int, name string, age short, salary double"

data_list = [(100, "Prashant", 45, 45000),(101, "Tarun", 36, 33000),(102, "David", 48, 28000)]

sample_df = spark.createDataFrame(data= data_list, schema = schema)

####2. Add following columns to your Dataframe
1. increment: 10% of the salary up to 3000 maximum increment
2. revised_salary: salary + increment

In [0]:
from pyspark.sql.functions import expr

salary_df = (
    sample_df.withColumns({
        "increment": expr("case when salary > 30000 then 3000 else salary * 10/100 end"),
        "revised_salary": expr("salary + increment")
        })
)

salary_df.display()

####3. Add following columns to your Dataframe
 * increment: 10% of the salary up to 3000 maximum increment

 Replace the following column in your dataframe
 * salary: current salary + increment

In [0]:
from pyspark.sql.functions import expr

salary_df = (
    sample_df.withColumn("increment", expr("case when salary > 30000 then 3000 else salary * 10/100 end"))
    .withColumn("salary", expr("salary+increment"))
)
salary_df.display()

####4. Add a batch number (uuid) column to your dataframe

In [0]:
import uuid
from pyspark.sql.functions import lit

batch_id = str(uuid.uuid4())
#uuid is a unique identifier which can be used to identify a batch of data
salary_batch_df = sample_df.withColumn("batch_id", lit(batch_id))
#lit is used instead of expr as lit is used to create a literal column. Expr only works when the column is already present in the dataframe

salary_batch_df.display()

####4. Rename the dataframe colums as listed below
 1. increment - annual_increment
 2. salary - incremented_salary

In [0]:
new_salary_df = (
    salary_df.withColumnsRenamed({
        "increment":"annual_increment",
        "salary":"incremented_salary"
    })
)
new_salary_df.display()

####5. Remove the following colums from your dataframe
 1. age
 2. annual_increment

In [0]:
small_salary_df = new_salary_df.drop("age","annual_increment")

small_salary_df.display()


###Dataframe Column Expressions
----------------------------------------------------------------------------------------------------------------------------------------

####Requirement
 Read data from flight_time and transform as below
 1. Rename fl_date to dep_date
 2. Compute arr_date
 3. Following fields to represent full timestamp
     1. crs_dep_time
     2. dep_time
     3. crs_arr_time
     4. arr_time

In [0]:
%sql
select * from dev.spark_db.flight_time limit 3

 1. Can we do it using select() or selectExpr() transformations?

In [0]:
flight_time_df = spark.read.table("dev.spark_db.flight_time")

flight_time_1_df = (
    flight_time_df.selectExpr(
        "fl_date as dep_date",
        "to_date(dep_date + dep_time + wheels_on + taxi_in) as arr_date",
        "dep_date + crs_dep_time as crs_dep_time",
        "dep_date + dep_time as dep_time",
        "arr_date + crs_arr_time as crs_arr_time",
        "arr_date + arr_time as arr_time",
    )
)
flight_time_1_df.where("op_carrier_fl_num = 1451 and dep_date = '2000-01-01'").display()

#selectExpr only showcases the columns that are selected and not the entire columns which might be present in the dataframe


2. Can we do it using withColumn() or withColumns()?

In [0]:
from pyspark.sql.functions import expr

flight_time_2_df = (
    flight_time_df.withColumnRenamed("fl_date", "dep_date")
    .withColumn("arr_date", expr("to_date(dep_date +dep_time + wheels_on + taxi_in) as arr_date"))
    .withColumns({
       "crs_dep_time": expr("dep_date + crs_dep_time"),
        "dep_time": expr("dep_date + dep_time"),
        "crs_arr_time": expr("arr_date + crs_arr_time"),
        "arr_time": expr("arr_date + arr_time")
    })
)
flight_time_2_df.where("op_carrier_fl_num = 1451 and dep_date = '2000-01-01'").display()

3. Alternative approach to write expressions\
  Why to use it?
    * It gives access to column functions

In [0]:
from pyspark.sql.functions import to_date, col

flight_time_3_df = (
    flight_time_df.withColumnRenamed("fl_date", "dep_date")
    .withColumn("arr_date", to_date(col("dep_date") +col("dep_time") + col("wheels_on") + col("taxi_in")))
    .withColumns({
       "crs_dep_time": col("dep_date") + col("crs_dep_time"),
        "dep_time": col("dep_date") + col("dep_time"),
        "crs_arr_time": col("arr_date") + col("crs_arr_time"),
        "arr_time": col("arr_date") + col("arr_time")
    })
)
flight_time_3_df.where((col("op_carrier_fl_num") == 1451) & (col("dep_date") == '2000-01-01')).display()

###Filtering and removing duplicates
----------------------------------------------------------------------------------------------------------------------------------------------

%md
####Requirement
Create a test Dataframe as shown below
```
  +---+---------+-----------+--------+
  | id|   source|destination|distance|
  +---+---------+-----------+--------+
  |101|   Mumbai|        Goa|     587|
  |102|   Mumbai|  Bangalore|     985|
  |102|   Mumbai|  Bangalore|     985|
  |103|    Delhi|    Chennai|    2208|
  |104|    Delhi|    Chennai|    2208|
  |105|Bangalore|    Kolkata|    1868|
  |105|Bangalore|    Kolkata|    1865|
  +---+---------+-----------+--------+
```

####2. Select records that satisfy the follwing criteria
 1. Source is Mumbai and destination is Bangalore

 Show the following approaches
 1. Using a single filter
 2. Using two filters
 3. Using a col function expression
 4. Using a dataframe qualifier expression

In [0]:
data_schema = "id int, source string, destination string, distance int"

data_list = [(101,"Mumbai","Goa",587),
             (102,"Mumbai","Bangalore",985),
             (102,"Mumbai","Bangalore",985),
             (103,"Delhi","Chennai",2208),
             (104,"Delhi","Chennai",2208),
             (105, "Bangalore","Kolkata",1868),
             (105, "Bangalore","Kolkata",1865)]

df = spark.createDataFrame(data=data_list, schema= data_schema)

df.display()

2.1 Using a single filter

In [0]:
df.filter("source = 'Mumbai' and destination = 'Bangalore'").display()


2.2 Using two filters

In [0]:
df.filter("source ='Mumbai'").filter("destination = 'Bangalore'").display()

2.3 Using a col function expression

In [0]:
from pyspark.sql.functions import col

df.filter((col("source")=='Mumbai') & (col("destination")=='Bangalore')).display()

2.4 Using a dataframe qualifier expression

In [0]:
df.filter((df.source == 'Mumbai') & (df.destination=='Bangalore')).display()

####3. Remove duplicate records from your dataframe

3.1 what do you mean by duplicate?

In [0]:
df.display()

3.2 Remove all the duplicates based on all the column values

In [0]:
df.distinct().display()

3.3 Remove duplicates based on the specified column values

In [0]:
df.dropDuplicates(["id","source","destination"]).display()

###Sorting, Limiting and Collecting
----------------------------------------------------------------------------------------------------------------------------------------------

####1. Read data from flight_time table

In [0]:
flight_time_df = spark.read.table("dev.spark_db.flight_time")
flight_time_df.display()

####2. Find the top 3 most delayed flights on 2000-01-16 from AUS to ORD

 Collect the results into a list and display the output as the following.
 ```
     AA flight delayed by 5.0 minutes
     AA flight delayed by 2.0 minutes
     UA flight delayed by 2.0 minutes
 ```

In [0]:
from pyspark.sql.functions import expr

top3_df = (
    flight_time_df.where("fl_date = '2000-01-16' and origin = 'AUS' and dest = 'ORD'")
    .withColumn("delayed_arrival", expr("crs_dep_time - dep_time"))
    .orderBy("delayed_arrival", ascending=False)
    .limit(3)
    )
top3_list_of_rows = top3_df.collect()

#collect is an action which will give all the records from this dataframe to python list
#collect is not a good idea to use in production as it will give all the records to driver
#and driver can crash if the data is too big
#but it is useful for testing and debugging

In [0]:
top3_list_of_rows

2.2 Make a list of dictionary from a list of rows

In [0]:
top_3_list = [row.asDict() for row in top3_list_of_rows]

#row is a spark dataframe class which gives a method asDict to convert row to dictionary

In [0]:
top_3_list

2.3 Print the result in desired format

In [0]:
for i in top_3_list:
    print(i["OP_CARRIER"] + " flight delayed by " + str(i["delayed_arrival"].total_seconds()/60 ) + " minutes")

####2. Find the third most delayed flights on 2000-01-16 from AUS to ORD

In [0]:
from pyspark.sql.functions import expr

top3rd_df = (
    flight_time_df.where("fl_date = '2000-01-16' and origin = 'AUS' and dest = 'ORD'")
    .withColumn("delayed_arrival", expr("crs_dep_time - dep_time"))
    .orderBy("delayed_arrival", ascending=False)
    .limit(3)
    .offset(2) #offset is used to skip the first n records
    )

top3rd_df.display()

###Transforming Unstructured data
----------------------------------------------------------------------------------------------------------------------------------------------

####1. Read data from the apache-logs.txt file

1.1 Load and display the data

In [0]:
file_df = (
    spark.read.format("text")
    .load(path="/Volumes/dev/spark_db/datasets/spark_programming/data/apache-logs.txt")
)
file_df.display()

1.2 Print the schema

In [0]:
file_df.printSchema()

####2. Develop an strategy to extract the following fields
 1. ip_address: It is the IP address of the site visitor.
 2. visit_timestamp: It is the date and time of the site visit. Parse and format the timestamp to YYYY-MM-DD HH:MI:SS Z
 3. visit_resource: Which resource from our website was accessed
 4. referring_url: It is the clean URL of the referring website.

2.1 Develop a regular expression

In [0]:
# 83.149.9.216 - - [17/May/2015:10:05:03 +0000] "GET /presentations/logstash-monitorama-2013/images/kibana-search.png HTTP/1.1" 200 203023 "http://semicomplete.com/presentations/logstash-monitorama-2013/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/32.0.1700.77 Safari/537.36"

log_reg = r'^(\S+) (\S+) (\S+) \[([\w:/]+\s[+\-]\d{4})\] "(\S+) (\S+) (\S+)" (\d{3}) (\S+) "(\S+)" "([^"]*)'

## Parses an access log line: uses (\S+) for space-delimited tokens (IP/ids/size), \[...\] for timestamp, "(\S+) (\S+) (\S+)" for method/path/protocol, (\d{3}) for status code, and quoted groups ("...") to safely capture referrer and user-agent including spaces

2.2 Apply regular expression to parse the record

In [0]:
from pyspark.sql.functions import regexp_extract

logs_df = (
    file_df.select(
        regexp_extract("value",log_reg, 1).alias("ip_address"), #regex extract returns a column object & column object gives us bunch of functions that can be applied such as alias
        regexp_extract("value", log_reg, 4).alias("visit_timestamp"),
        regexp_extract("value", log_reg, 6).alias("visit_resource"),
        regexp_extract("value", log_reg, 10).alias("referring_url")
    )
)

logs_df.display()

2.3 Refine results with further transformations

In [0]:
from pyspark.sql.functions import to_timestamp, substring_index
test_df = (
    logs_df.withColumns({
        "visit_timestamp":to_timestamp("visit_timestamp","dd/MMM/yyyy:HH:mm:ss Z"), #here dd/MMM/yyyy:HH:mm:ss Z is the input timestamp format which will be convert to yyyy-MM-dd HH:mm:ss Z through to_timestamp function
        "referring_url": substring_index("referring_url","/",3)
    })
)

test_df.display()

### Transforming data with LLM
-------------------------------------------------------------------------------------------------------------------------------------------

####3. Use AI to parse and extract the required information from unstructured data

3.1 Prepare an AI prompt to extract the required information

In [0]:
prompt = """
You will be provided with an Apache log file record. It is an unstructured text record. 
Each record represents some information for our website visits, such as what is the IP address of the visitor, 
What is the date and time of the visit, which resource was requested, and the URL of the referring website? 
You are asked to parse the log file record and extract the following fields.
ip_address: It is the IP address of the site visitor.
visit_timestamp: It is the date and time of the site visit. Parse and format the timestamp to YYYY-MM-DD HH:MI:SS Z
visit_resource: Which resource from our website was accessed?
referring_url: It is the clean URL of the referring website. When the actual referring URL is not given, 
you can extract the URL from the user agent. For cleaning the URL, you should take the values only up to the domain extension, 
such as .com, .in, .uk, etc.
Give only the final answer in the JSON format.
Record:
"""

3.2 Develop an AI query expression

In [0]:
#Ai query function is a databrick feature not spark

from pyspark.sql.functions import col, concat, lit

result_df = (
    file_df.limit(10)
    .withColumn("prompt", concat(lit(prompt), col("value")))
    .withColumn("json_extract", expr("""
            ai_query(
                endpoint=> 'databricks-llama-4-maverick',
                request=> prompt,
                responseFormat=> 'struct<extract: struct<
                ip_address: string,
                visit_timestamp: string,
                visit_resource: string,
                referring_url: string
                >>')"""))
)

result_df.display()

3.3 Parse the JSON extract to individual columns

In [0]:
from pyspark.sql.functions import from_json, col, to_timestamp
extract_schema = "ip_address string, visit_timestamp string, visit_resource string, referring_url string"
final_result_df = (
    result_df.withColumn("json_extract", from_json(col("json_extract"),extract_schema)) #from_json function will convert the JSON string to struct
    .selectExpr("json_extract.*") #select all columns from json_extract
    .withColumn("visit_timestamp", to_timestamp("visit_timestamp","yyyy-MM-dd HH:mm:ss Z"))
)

final_result_df.display()